In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Tạo folder chính trong Drive
BASE_DRIVE = '/content/drive/MyDrive/ISIC_YOLO'
os.makedirs(BASE_DRIVE, exist_ok=True)

print("Drive đã mount và folder sẵn sàng!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive đã mount và folder sẵn sàng!


In [8]:
# Cài Kaggle
!pip install -q kaggle

# Tạo thư mục và copy kaggle.json
!mkdir -p ~/.kaggle

# Copy từ Drive (đường dẫn bạn vừa tạo)
!cp "/content/drive/MyDrive/Kaggle/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle API đã thiết lập!")

cp: cannot stat '/content/drive/MyDrive/Kaggle/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
✅ Kaggle API đã thiết lập!


In [9]:
raw_data_path = f"{BASE_DRIVE}/raw_data"

if os.path.exists(raw_data_path) and len(os.listdir(raw_data_path)) > 0:
    print("✅ Dataset đã tồn tại trong Drive, bỏ qua tải mới.")
else:
    print("Đang tải dataset...")
    !kaggle datasets download -d tschandl/isic2018-challenge-task1-data-segmentation -p {BASE_DRIVE}
    !unzip -q {BASE_DRIVE}/isic2018-challenge-task1-data-segmentation.zip -d {raw_data_path}

# Kiểm tra cấu trúc và tên mask
print("\nCấu trúc thư mục:")
!ls {raw_data_path}

mask_dir = f"{raw_data_path}/ISIC2018_Task1_Training_GroundTruth"
print("\n5 tên file mask mẫu:")
!ls "{mask_dir}" | head -10

Đang tải dataset...
Dataset URL: https://www.kaggle.com/datasets/tschandl/isic2018-challenge-task1-data-segmentation
License(s): CC0-1.0
100% 12.9G/12.9G [10:28<00:00, 22.0MB/s]


Cấu trúc thư mục:
ISIC2018_Task1-2_Test_Input	 ISIC2018_Task1-2_Validation_Input
ISIC2018_Task1-2_Training_Input  ISIC2018_Task1_Training_GroundTruth

5 tên file mask mẫu:
ATTRIBUTION.txt
ISIC_0000000_segmentation.png
ISIC_0000001_segmentation.png
ISIC_0000003_segmentation.png
ISIC_0000004_segmentation.png
ISIC_0000006_segmentation.png
ISIC_0000007_segmentation.png
ISIC_0000008_segmentation.png
ISIC_0000009_segmentation.png
ISIC_0000011_segmentation.png


In [10]:
# Kiểm tra cấu trúc
!ls {BASE_DRIVE}/raw_data

# Xem 5 tên file mask (để biết mask có đuôi gì)
mask_dir = f"{BASE_DRIVE}/raw_data/ISIC2018_Task1_Training_GroundTruth"
print("\n5 tên file mask mẫu:")
!ls {mask_dir} | head -5

ISIC2018_Task1-2_Test_Input	 ISIC2018_Task1-2_Validation_Input
ISIC2018_Task1-2_Training_Input  ISIC2018_Task1_Training_GroundTruth

5 tên file mask mẫu:
ATTRIBUTION.txt
ISIC_0000000_segmentation.png
ISIC_0000001_segmentation.png
ISIC_0000003_segmentation.png
ISIC_0000004_segmentation.png


Tổng số ảnh gốc: 2594
Train: 1815 ảnh
Val:   389 ảnh
Test:  390 ảnh


Xử lý test: 100%|██████████| 390/390 [00:00<00:00, 5649.03it/s]

✅ Hoàn tất split và tạo label YOLO!


In [14]:
# Generate YOLO Label

train_img_dir = f"{BASE_DRIVE}/raw_data/ISIC2018_Task1-2_Training_Input"
train_mask_dir = f"{BASE_DRIVE}/raw_data/ISIC2018_Task1_Training_GroundTruth"

dataset_dir = f"{BASE_DRIVE}/dataset_yolo"

# Xóa dataset cũ để làm lại sạch (nếu muốn)
# shutil.rmtree(dataset_dir, ignore_errors=True)

for split in ['train', 'val', 'test']:
    os.makedirs(f"{dataset_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{dataset_dir}/labels/{split}", exist_ok=True)

image_files = [f for f in os.listdir(train_img_dir) if f.endswith('.jpg')]

train_files, temp = train_test_split(image_files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp, test_size=0.5, random_state=42)

print(f"Tổng ảnh: {len(image_files)} → Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

def generate_yolo_label(mask_path, img_shape):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)

    img_h, img_w = img_shape[:2]
    x_center = np.clip((x + w/2) / img_w, 0, 1)
    y_center = np.clip((y + h/2) / img_h, 0, 1)
    width    = np.clip(w / img_w, 0, 1)
    height   = np.clip(h / img_h, 0, 1)

    return f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

# ====================== SỬA TÊN MASK Ở ĐÂY ======================
def process_split(files, split):
    success = 0
    for img_file in tqdm(files, desc=f"Xử lý {split}"):
        base_name = img_file.replace('.jpg', '')

        # Tên mask cố định là _segmentation
        mask_filename = f"{base_name}_segmentation.png"
        mask_path = os.path.join(train_mask_dir, mask_filename)

        if not os.path.exists(mask_path):
            continue  # Bỏ qua nếu không tìm thấy mask

        img_path = os.path.join(train_img_dir, img_file)

        # Copy ảnh
        shutil.copy(img_path, f"{dataset_dir}/images/{split}/{img_file}")

        # Tạo label
        img = cv2.imread(img_path)
        label = generate_yolo_label(mask_path, img.shape)
        if label:
            label_path = f"{dataset_dir}/labels/{split}/{base_name}.txt"
            with open(label_path, 'w') as f:
                f.write(label + '\n')
            success += 1

    print(f"✅ {split}: Tạo thành công {success}/{len(files)} labels")

# Chạy xử lý lại
process_split(train_files, 'train')
process_split(val_files, 'val')
process_split(test_files, 'test')


Tổng ảnh: 2594 → Train: 1815 | Val: 389 | Test: 390


Xử lý train: 100%|██████████| 1815/1815 [08:39<00:00,  3.50it/s]


✅ train: Tạo thành công 1815/1815 labels


Xử lý val: 100%|██████████| 389/389 [09:12<00:00,  1.42s/it]


✅ val: Tạo thành công 389/389 labels


Xử lý test: 100%|██████████| 390/390 [09:57<00:00,  1.53s/it]

✅ test: Tạo thành công 390/390 labels


In [15]:
print("=== THỐNG KÊ DATASET ===")
print(f"Số class          : 1 (lesion)")
print(f"Train images      : {len(train_files)}")
print(f"Validation images : {len(val_files)}")
print(f"Test images       : {len(test_files)}")
print(f"Tổng cộng         : {len(train_files) + len(val_files) + len(test_files)} ảnh")

=== THỐNG KÊ DATASET ===
Số class          : 1 (lesion)
Train images      : 1815
Validation images : 389
Test images       : 390
Tổng cộng         : 2594 ảnh


In [18]:
import math # Add this import

def plot_samples(num_samples=6):
    train_label_dir = f"{dataset_dir}/labels/train"
    train_img_dir   = f"{dataset_dir}/images/train"

    label_files = [f for f in os.listdir(train_label_dir) if f.endswith('.txt')]
    # Ensure we don't try to plot more files than available
    label_files_to_plot = label_files[:num_samples]

    if not label_files_to_plot:
        print("Không tìm thấy file nhãn nào để hiển thị.")
        return

    # Calculate rows and cols for subplots dynamically
    cols = min(len(label_files_to_plot), 5) # Max 5 columns for better layout
    rows = math.ceil(len(label_files_to_plot) / cols) if cols > 0 else 1

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axs = axs.ravel() # Flatten the 2D array of axes for easier iteration

    for i, txt_file in enumerate(label_files_to_plot):
        base = txt_file.replace('.txt', '')
        img_path = os.path.join(train_img_dir, base + '.jpg')
        label_path = os.path.join(train_label_dir, txt_file)

        img = cv2.imread(img_path)
        if img is None:
            print(f"Cảnh báo: Không thể đọc ảnh {img_path}. Bỏ qua.")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Đọc bbox
        with open(label_path, 'r') as f:
            line = f.readline().strip().split()
            if len(line) < 5:
                print(f"Cảnh báo: Định dạng nhãn không hợp lệ trong {label_path}. Bỏ qua.")
                continue
            _, x_c, y_c, w, h = map(float, line)

        h_img, w_img = img.shape[:2]
        x = int((x_c - w/2) * w_img)
        y = int((y_c - h/2) * h_img)
        w = int(w * w_img)
        h = int(h * h_img)

        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 4)
        axs[i].imshow(img)
        axs[i].axis('off')
        axs[i].set_title(f"Ảnh {base}")

    # Hide any unused subplots
    for j in range(len(label_files_to_plot), len(axs)):
        fig.delaxes(axs[j])

    plt.tight_layout()
    plt.show()


In [19]:
plot_samples(100)
print("✅ Đã hiển thị 100 ảnh train kèm bounding box (màu đỏ)")

Output hidden; open in https://colab.research.google.com to view.